# Week 7 – Model Evaluation & Feature Engineering
## Day 1: Cross-Validation (k-fold)

A single train/test split can be lucky or unlucky. K-fold cross-validation splits the data into k parts, trains on k-1, tests on the remaining 1, and repeats — giving a more reliable estimate of how a model actually performs.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin",
        "BMI","DiabetesPedigreeFunction","Age","Outcome"]
df = pd.read_csv(url, names=cols)

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [2]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(dt, X, y, cv=kfold, scoring="f1")

print("F1 scores across 5 folds:", cv_scores)
print("Mean F1:", cv_scores.mean(), " Std:", cv_scores.std())

F1 scores across 5 folds: [0.68627451 0.58426966 0.63366337 0.65934066 0.63461538]
Mean F1: 0.6396327166035894  Std: 0.0337334681163451


In [3]:
dt.fit(X_train, y_train)
single_split_f1 = f1_score(y_test, dt.predict(X_test))

print("Single train/test split F1:", single_split_f1)
print("5-fold CV mean F1:", cv_scores.mean())

Single train/test split F1: 0.7037037037037037
5-fold CV mean F1: 0.6396327166035894


### Observation
[Fill in after running: is the single-split score higher or lower than the CV mean? What does that tell you about whether the original Week 6 split was lucky or unlucky?]

## Day 2: Feature Engineering
This dataset is all numeric (no categorical columns to encode), so feature engineering here focuses on scaling and handling outliers/invalid values.

In [4]:
zero_invalid_cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
for col in zero_invalid_cols:
    print(col, "zeros:", (df[col] == 0).sum())

Glucose zeros: 5
BloodPressure zeros: 35
SkinThickness zeros: 227
Insulin zeros: 374
BMI zeros: 11


In [5]:
df_clean = df.copy()
for col in zero_invalid_cols:
    median_val = df_clean.loc[df_clean[col] != 0, col].median()
    df_clean[col] = df_clean[col].replace(0, median_val)

df_clean.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,121.656250,72.386719,29.108073,140.671875,32.455208,0.471876,33.240885,0.348958
std,3.369578,30.438286,12.096642,8.791221,86.383060,6.875177,0.331329,11.760232,0.476951
min,0.000000,44.000000,24.000000,7.000000,14.000000,18.200000,0.078000,21.000000,0.000000
25%,1.000000,99.750000,64.000000,25.000000,121.500000,27.500000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,29.000000,125.000000,32.300000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000
